In [1]:
!pip install openmeteo-requests retry-requests requests-cache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.1/167.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.2/683.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.1 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.2.10
    Uninstalling flatbuffers-25.2.10:
      Successfully uninstalled flatbuffers-25.2.10
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.

In [2]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import matplotlib.pyplot as plt
import os
from datetime import datetime, timedelta

def setup_client():
    """
    Menyiapkan client API dengan fitur Cache dan Retry.
    Tujuannya agar koneksi stabil dan hemat kuota.
    """
    # Buat cache bernama '.cache', data disimpan selama 3600 detik (1 jam)
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)

    # Jika gagal connect, coba lagi (retry) sampai 5x
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

    # Return objek client yang siap pakai
    return openmeteo_requests.Client(session=retry_session)

In [3]:
# ==========================================
# 2. FETCH DATA PER CHUNK
# ==========================================
def fetch_chunk(client, lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "rain",
            "wind_speed_10m",
            "wind_direction_10m",
            "surface_pressure",
            "weather_code"
            ],
        "timezone": "Asia/Jakarta"
    }

    print(f"   ⏳ Mengambil: {start_date} s.d {end_date}...")
    try:
        responses = client.weather_api(url, params=params)
        return process_data(responses[0]) # Langsung olah jadi DataFrame
    except Exception as e:
        print(f"   ❌ Gagal pada chunk ini: {e}")
        return None

# ==========================================
# 3. PROCESS DATA (SAMA SEPERTI SEBELUMNYA)
# ==========================================
def process_data(response):
    hourly = response.Hourly()

    date_range = pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq = pd.Timedelta(seconds=hourly.Interval()),
        inclusive = "left"
    )

    df = pd.DataFrame(data = {
        "date": date_range,
        "temperature": hourly.Variables(0).ValuesAsNumpy(),
        "humidity": hourly.Variables(1).ValuesAsNumpy(),
        "dewpoint":hourly.Variables(2).ValuesAsNumpy(),
        "rain_mm": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed": hourly.Variables(4).ValuesAsNumpy(),
        "wind_direction": hourly.Variables(5).ValuesAsNumpy(),
        "pressure": hourly.Variables(6).ValuesAsNumpy(),
        "weather_code": hourly.Variables(7).ValuesAsNumpy()
    })

    df = df.set_index('date')
    df.index = df.index.tz_convert('Asia/Jakarta')
    return df

# ==========================================
# 4. LOGIKA UTAMA: CHUNKING & MERGING
# ==========================================
def fetch_long_period_data(client, lat, lon, start_str, end_str, folder_tujuan, chunk_years=10):

    if not os.path.exists(folder_tujuan):
        os.makedirs(folder_tujuan)

    start_date = datetime.strptime(start_str, "%Y-%m-%d")
    final_end_date = datetime.strptime(end_str, "%Y-%m-%d")

    all_files = []

    current_start = start_date

    print(f"🚀 Memulai Misi Pengambilan Data 75 Tahun ({chunk_years} tahunan)...")

    while current_start < final_end_date:
        # Hitung tanggal akhir chunk ini
        # Misal start 1950, tambah 10 tahun -> 1960. Dikurang 1 hari biar gak overlap.
        current_end = current_start.replace(year=current_start.year + chunk_years) - timedelta(days=1)

        # Jika current_end melebihi batas akhir request, pakai batas akhir request
        if current_end > final_end_date:
            current_end = final_end_date

        # Format string untuk API
        s_str = current_start.strftime("%Y-%m-%d")
        e_str = current_end.strftime("%Y-%m-%d")

        # Nama file sementara
        chunk_filename = os.path.join(folder_tujuan, f"temp_{s_str}_{e_str}.csv")

        # Cek apakah file sudah ada? (Resume Capability)
        if os.path.exists(chunk_filename):
            print(f"⏩ Skip: {s_str} - {e_str} (Sudah ada)")
            all_files.append(chunk_filename)
        else:
            # Ambil Data
            df_chunk = fetch_chunk(client, lat, lon, s_str, e_str)
            if df_chunk is not None:
                df_chunk.to_csv(chunk_filename)
                print(f"   ✅ Tersimpan: {chunk_filename}")
                all_files.append(chunk_filename)
            else:
                print("   ⚠️ Chunk ini dilewati karena error.")

        # Lanjut ke periode berikutnya
        current_start = current_end + timedelta(days=1)

    print("\n🔗 Menggabungkan semua pecahan data...")

    # Gabungkan semua CSV jadi satu
    df_list = []
    for f in all_files:
        df = pd.read_csv(f, index_col='date', parse_dates=True)
        df_list.append(df)

    if df_list:
        df_final = pd.concat(df_list)
        df_final = df_final.sort_index() # Urutkan waktu biar rapi
        # Hapus duplikat jika ada irisan
        df_final = df_final[~df_final.index.duplicated(keep='first')]

        print(f"🎉 SUKSES BESAR! Total Data: {len(df_final)} baris.")
        print(f"   Mulai: {df_final.index.min()}")
        print(f"   Akhir: {df_final.index.max()}")
        return df_final
    else:
        return None

In [4]:
import time
from datetime import date, timedelta
import os
# Pastikan library lain yang dibutuhkan (openmeteo_requests, pandas, retry, dll) sudah terimport di bagian atas filemu

# ==========================================
# FUNGSI TAMBAHAN: RETRY LOGIC (ANTI GAGAL)
# ==========================================
def fetch_chunk_safe(func_fetch, *args, max_retries=5, **kwargs):
    """
    Mencoba menjalankan fungsi fetch. Jika gagal (kena limit API),
    akan menunggu dan mencoba lagi secara otomatis.
    """
    wait_seconds = 60 # Tunggu 1 menit jika gagal (sesuai aturan Open Meteo)

    for attempt in range(1, max_retries + 1):
        try:
            return func_fetch(*args, **kwargs)
        except Exception as e:
            print(f"⚠️ Percobaan ke-{attempt} gagal: {e}")
            if attempt < max_retries:
                print(f"⏳ Menunggu {wait_seconds} detik sebelum mencoba lagi...")
                time.sleep(wait_seconds)
            else:
                print("❌ Gagal total setelah beberapa kali percobaan.")
                raise e # Lempar error jika sudah menyerah

# ==========================================
# EKSEKUSI UTAMA
# ==========================================
if __name__ == "__main__":
    # 1. Konfigurasi Lokasi
    # Perhatikan: tuple koma di akhir LAT dihapus agar aman jadi float biasa
    LAT = -7.736436737566032
    LON = 109.6460550796716

    # 2. Konfigurasi Waktu (OTOMATIS)
    MULAI = "1950-01-01"

    # Hitung tanggal kemarin (H-1) secara otomatis
    kemarin = date.today() - timedelta(days=1)
    AKHIR = kemarin.strftime("%Y-%m-%d")

    print(f"📅 Jadwal Run Otomatis: Mengambil data dari {MULAI} sampai {AKHIR}")

    # 3. Konfigurasi File
    FOLDER = "open_meteo_climate"
    FILE_FINAL = "kebumen_75tahun_lengkap.csv"

    # Setup Client (Pastikan fungsi setup_client() ada di kodemu sebelumnya)
    client = setup_client()

    # 4. Jalankan Fetching dengan Pengaman (Safe Mode)
    # Kita bungkus proses fetching utama dalam try-except besar atau gunakan logika retry di level chunk
    # Namun, karena fetch_long_period_data sudah melakukan looping internal,
    # kita modifikasi sedikit cara panggilannya atau pastikan fungsi fetch_long_period_data
    # memiliki jeda (sleep) antar request.

    try:
        # Jalankan fungsi utamamu
        # Tips: Open-Meteo membatasi request per menit.
        # Pastikan di dalam fungsi `fetch_long_period_data` ada time.sleep(5) antar chunk.
        df_lengkap = fetch_long_period_data(client, LAT, LON, MULAI, AKHIR, FOLDER, chunk_years=5)

        if df_lengkap is not None:
            # Simpan Hasil Akhir
            if not os.path.exists(FOLDER):
                os.makedirs(FOLDER)

            path_final = os.path.join(FOLDER, FILE_FINAL)
            df_lengkap.to_csv(path_final, index=False) # index=False biar rapi
            print(f"✅ SUKSES! File Gabungan Tersimpan: {path_final}")

            # Hapus file temp (Opsional)
            # import glob
            # for f in glob.glob(f"{FOLDER}/temp_*.csv"):
            #     os.remove(f)

    except Exception as e:
        print(f"❌ Terjadi kesalahan fatal pada script utama: {e}")

📅 Jadwal Run Otomatis: Mengambil data dari 1950-01-01 sampai 2025-12-06
🚀 Memulai Misi Pengambilan Data 75 Tahun (5 tahunan)...
   ⏳ Mengambil: 1950-01-01 s.d 1954-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1950-01-01_1954-12-31.csv
   ⏳ Mengambil: 1955-01-01 s.d 1959-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1955-01-01_1959-12-31.csv
   ⏳ Mengambil: 1960-01-01 s.d 1964-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1960-01-01_1964-12-31.csv
   ⏳ Mengambil: 1965-01-01 s.d 1969-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1965-01-01_1969-12-31.csv
   ⏳ Mengambil: 1970-01-01 s.d 1974-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1970-01-01_1974-12-31.csv
   ⏳ Mengambil: 1975-01-01 s.d 1979-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1975-01-01_1979-12-31.csv
   ⏳ Mengambil: 1980-01-01 s.d 1984-12-31...
   ❌ Gagal pada chunk ini: failed to request 'https://archive-api.open-meteo.com/v1/archive': {'error': True, 'reason': 'Minutely API request limit exceeded. Ple

In [5]:
data = pd.read_csv("/kaggle/working/open_meteo_climate/kebumen_75tahun_lengkap.csv")
data.tail(10)

,temperature,humidity,dewpoint,rain_mm,wind_speed,wind_direction,pressure,weather_code
262958,28.245,79.906044,24.445,0.2,9.000000,270.00000,1004.43490,51.0
262959,27.995,81.321390,24.495,0.1,11.542478,266.42374,1003.83430,51.0
262960,25.795,86.368510,23.345,5.4,11.165805,271.84756,1004.01807,63.0
262961,25.645,86.354300,23.195,0.7,8.557102,284.62090,1004.91490,53.0
262962,25.295,88.703070,23.295,0.2,5.804825,277.12490,1005.51110,51.0
262963,25.095,91.956940,23.695,0.0,4.735060,261.25390,1006.20830,3.0
262964,24.495,93.886140,23.445,0.0,5.411986,266.18600,1007.20170,3.0
262965,24.145,94.439800,23.195,0.0,5.400000,270.00000,1007.69790,3.0
262966,24.095,95.010220,23.245,0.0,5.771239,266.42374,1007.89703,3.0
262967,23.995,93.297090,22.845,0.0,7.200000,270.00000,1007.49730,3.0


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262968 entries, 0 to 262967
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   temperature     262968 non-null  float64
 1   humidity        262968 non-null  float64
 2   dewpoint        262968 non-null  float64
 3   rain_mm         262968 non-null  float64
 4   wind_speed      262968 non-null  float64
 5   wind_direction  262968 non-null  float64
 6   pressure        262968 non-null  float64
 7   weather_code    262968 non-null  float64
dtypes: float64(8)
memory usage: 16.1 MB
